# QM 640 Capstone — Step 3b: CIK-to-Ticker Mapping

SEC EDGAR identifies filers by CIK (Central Index Key), not stock ticker.
Yahoo Finance needs a ticker. This pulls the SEC's own free CIK-ticker
mapping file and merges it into your screening worksheet.

**Run this after Part A of Step 3 (worksheet build) and before Step 4
(Yahoo Finance pull).**

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every
other cell in this notebook reads from and writes to.

In [1]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "Shan_muganathan@yahoo.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 931, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 931 (delta 52), reused 79 (delta 30), pack-reused 813 (from 1)
Receiving objects: 100% (931/931), 6.65 MiB | 20.15 MiB/s, done.
Resolving deltas: 100% (493/493), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies

In [2]:
!pip install -q pandas requests

## Cell 3 — Configuration

In [3]:
import os

HEADERS = {"User-Agent": "QM640 Capstone research Shan_muganathan@yahoo.com"}
RAW_DIR = os.path.join(BASE_DIR, "data/raw")

## Cell 4 — Pull the SEC's CIK-ticker crosswalk and merge it in

In [4]:
import pandas as pd
import requests


def get_cik_ticker_map():
    """SEC's own free CIK<->ticker crosswalk, updated regularly."""
    url = "https://www.sec.gov/files/company_tickers.json"
    resp = requests.get(url, headers=HEADERS, timeout=20)
    resp.raise_for_status()
    data = resp.json()

    df = pd.DataFrame.from_dict(data, orient="index")
    df = df.rename(columns={"cik_str": "cik", "title": "company_name"})
    df["cik"] = df["cik"].astype(int)

    n_before = len(df)
    n_ciks = df["cik"].nunique()

    # SEC's crosswalk has ONE ROW PER TICKER, not per company - a firm with
    # common stock plus several preferred-share series (e.g. large banks) has
    # multiple rows under the same CIK. Merging on CIK alone without deduping
    # this first fans out every filing for that firm into one row per ticker -
    # this is exactly what took a 9,419-event worksheet to 11,676 rows earlier.
    # Keep exactly one CANONICAL ticker per CIK: prefer tickers without a
    # hyphen (SEC uses "-P"/"-WS" suffixes for preferred shares/warrants, so a
    # bare ticker is almost always the common stock), then the shortest, then
    # alphabetically first, as a stable tiebreak.
    df["is_preferred_or_special"] = df["ticker"].astype(str).str.contains("-", na=False)
    df["ticker_len"] = df["ticker"].astype(str).str.len()
    df = df.sort_values(["cik", "is_preferred_or_special", "ticker_len", "ticker"])
    df = df.drop_duplicates(subset=["cik"], keep="first")
    df = df.drop(columns=["is_preferred_or_special", "ticker_len"])

    out_path = os.path.join(RAW_DIR, "cik_ticker_map.csv")
    df.to_csv(out_path, index=False)
    print(f"CIK-ticker map: {n_before} raw entries ({n_ciks} unique CIKs) -> "
          f"deduplicated to {len(df)} rows (one canonical ticker per CIK) -> {out_path}")
    return df


def merge_into_screening_worksheet():
    cik_map = get_cik_ticker_map()
    worksheet_path = os.path.join(RAW_DIR, "screening_worksheet.csv")
    worksheet = pd.read_csv(worksheet_path)

    # Safe to re-run: if a previous run already merged a ticker column in,
    # drop it first so this merge can't collide and produce ticker_x/ticker_y
    # instead of a clean "ticker" column - this is exactly what broke on re-run.
    if "ticker" in worksheet.columns:
        print("Existing 'ticker' column found from a previous run - dropping and re-merging fresh.")
        worksheet = worksheet.drop(columns=["ticker"])

    n_before = len(worksheet)
    worksheet = worksheet.merge(cik_map[["cik", "ticker"]], on="cik", how="left")

    # Fail loud instead of silently fanning out rows - this is exactly the
    # bug that inflated the worksheet earlier. A left merge on a CIK-unique
    # cik_map can only ever match 1:1 or produce a NaN, never multiply rows.
    assert len(worksheet) == n_before, (
        f"Merge fanned out rows: {n_before} -> {len(worksheet)}. "
        f"cik_ticker_map.csv must have exactly one row per CIK - check the "
        f"dedup step in get_cik_ticker_map() above before re-running."
    )

    unmatched = worksheet["ticker"].isna().sum()
    if unmatched:
        print(f"Warning: {unmatched} events have no ticker match - these firms "
              f"may not be on the SEC's mapping (e.g., recently delisted or "
              f"foreign private issuers). Review manually before excluding.")

    worksheet.to_csv(worksheet_path, index=False)
    print(f"Merged ticker into {worksheet_path} ({len(worksheet)} rows, unchanged from {n_before})")
    return worksheet


worksheet = merge_into_screening_worksheet()
worksheet.head()

CIK-ticker map: 10432 raw entries (8017 unique CIKs) -> deduplicated to 8017 rows (one canonical ticker per CIK) -> /content/QM640-WALSH-CAPSTONE/data/raw/cik_ticker_map.csv
Existing 'ticker' column found from a previous run - dropping and re-merging fresh.
Merged ticker into /content/QM640-WALSH-CAPSTONE/data/raw/screening_worksheet.csv (9419 rows, unchanged from 9419)


,query,cik,company_name,form_type,file_date,accession_no,adsh,file_name,is_genuine_ai_event,announcement_type,confounding_event_flag,trading_halt_flag,sufficient_history_flag,exclude_reason,filing_url,screener_notes,item_codes,item_in_scope,ticker
0,"""AI capabilities""",876167,PROGRESS SOFTWARE CORP /MA (PRGS) (CIK 00008...,8-K,2023-01-03,0000876167-23-000004:pressrelease-marklogic.htm,0000876167-23-000004,NaN,NaN,NaN,N,REVIEW,N,insufficient pre-event price history (<120 tra...,https://www.sec.gov/cgi-bin/browse-edgar?actio...,NaN,"1.01,7.01,9.01",Y,PRGS
1,"""AI-powered""",1013857,PEGASYSTEMS INC (PEGA) (CIK 0001013857),8-K,2023-01-03,0001193125-23-000843:d442682dex991.htm,0001193125-23-000843,NaN,NaN,NaN,N,REVIEW,N,insufficient pre-event price history (<120 tra...,https://www.sec.gov/cgi-bin/browse-edgar?actio...,NaN,"2.05,7.01,9.01",N,PEGA
2,"""AI-powered""",1293818,OPGEN INC (OPGN) (CIK 0001293818),8-K,2023-01-04,0001079973-23-000010:ex99x1.htm,0001079973-23-000010,NaN,NaN,NaN,Y,REVIEW,N,confounding filing within +/-2 days; insuffici...,https://www.sec.gov/cgi-bin/browse-edgar?actio...,NaN,"3.03,5.03,8.01,9.01",Y,CFOR
3,"""AI-based""",278165,OMNIQ Corp. (OMQS) (CIK 0000278165),8-K,2023-01-04,0001493152-23-000229:ex99-1.htm,0001493152-23-000229,NaN,NaN,NaN,N,REVIEW,N,insufficient pre-event price history (<120 tra...,https://www.sec.gov/cgi-bin/browse-edgar?actio...,NaN,"7.01,9.01",N,OMQS
4,"""AI-driven""",1758766,"STEM, INC. (STEM) (CIK 0001758766)",8-K,2023-01-05,0001758766-23-000003:exhibit99-1320238xkgsconf...,0001758766-23-000003,NaN,NaN,NaN,N,REVIEW,N,insufficient pre-event price history (<120 tra...,https://www.sec.gov/cgi-bin/browse-edgar?actio...,NaN,"7.01,9.01",N,STEM


## Commit and push results back to GitHub

In [5]:
!git -C {BASE_DIR} add "data/raw/cik_ticker_map.csv"
!git -C {BASE_DIR} add "data/raw/screening_worksheet.csv"
!git -C {BASE_DIR} commit -m "Step 3b: merge CIK-to-ticker mapping into screening worksheet"
!git -C {BASE_DIR} push

[main 7a0c5c7] Step 3b: merge CIK-to-ticker mapping into screening worksheet
 2 files changed, 17308 insertions(+), 19723 deletions(-)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (6/6), 174.83 KiB | 1.71 MiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE.git
   7e0b13a..7a0c5c7  main -> main
